In [1]:
from rich.console import Console
from dotenv import load_dotenv
import os
import json
from openai import OpenAI
import gradio as gr

In [2]:
load_dotenv(override=True)
openai = OpenAI()

def show(text):
    import re
    clean = re.sub(r'\[/?[^\]]+\]', '', str(text))
    print(clean, flush=True)

In [3]:
todos =[]
completed = []

In [4]:
def get_todo_report(silent=False) -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index+1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index+1}: {todo}\n"
    if not silent:
        show(result)
    return result

In [5]:
def create_todos(descriptions: list[str]) -> str:
    if todos:
        return get_todo_report(silent=True)
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    show(f"Created {len(descriptions)} todos")
    return get_todo_report(silent=True)

In [6]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    show(completion_notes)
    return get_todo_report(silent=True)

In [7]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                "type": "array",
                "items": {"type": "string"}
            }
        },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "index": {
                "type": "integer",
                "description": "The 1-based index of the todo to mark as complete"
            },
            "completion_notes": {
                "type": "string",
                "description": "Notes about how you completed the todo"
            }
        },
        "required": ["index", "completion_notes"],
        "additionalProperties": False
    }
}

In [8]:
tools = [{"type": "function", "function": create_todos_json},
         {"type": "function", "function": mark_complete_json}]

In [9]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id
        })
    return results

In [10]:
system_message = """
You are given a problem to solve using your todo tools.
IMPORTANT RULES:
1. Call create_todos ONLY ONCE at the beginning to plan all steps
2. Then call mark_complete for each step in order
3. Do NOT call create_todos again after planning
4. After all todos are marked complete, provide your final answer
Do not ask the user questions. Respond only with the answer.
"""

In [11]:
import re

def strip_rich_markup(text):
    return re.sub(r'\[/?[^\]]+\]', '', text)

In [12]:
def chat(message, history):
    todos.clear()
    completed.clear()
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:
        response = openai.chat.completions.create(
            model="gpt-5-nano", messages=messages, tools=tools)
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    
    # Add todo report to the final answer
    todo_report = strip_rich_markup(get_todo_report(silent=False))
    final_answer = strip_rich_markup(response.choices[0].message.content)
    return f"**📋 Plan:**\n\n{todo_report}\n\n**✅ Answer:**\n\n{final_answer}"

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Tool called: create_todos
Created 7 todos
Tool called: mark_complete
Completed Option A cash-flow model plan: revenue_t = 12000*(1.15)^(t-1). Expenses = base salaries 17k + rent 3k + two new developers 16k = 36k per month. Start cash = 85k. Prepared month-by-month table (1-12) for burn and cash balance.
Tool called: mark_complete
Option A: Cash balance hits zero during Month 5. End of Month 4 balance ~ $0.92k; Month 5 net cash flow ~ -$15.01k, causing exhaustion within Month 5.
Tool called: mark_complete
Option A: Break-even would occur in month 10 if operations continued; since cash runs out in month 5, break-even is not achievable without external funding.
Todo #1: Build a cash-flow model for Option A: revenue grows 12k * 1.15^(t-1); expenses = base salaries (8+5+4=17k) + rent 3k = 20k + two additional devs at 8k each = 16k; so total expenses 36k; simulate month-by-month with starting cash 85k.
Todo #2: Determine the month when cash balance reaches zero or below under Option A; ident